In [108]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝 
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [109]:
df = pd.read_csv('data/merged_final_data.csv')

In [110]:
cols = df.columns
cols 

Index(['index', 'order_id', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'customer_lat', 'customer_lng', 'total_items_count', 'seller_id',
       'price', 'freight_value', 'review_score', 'category',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_lat',
       'seller_lng', 'order_purchase_dayofweek', 'order_purchase_month',
       'approved_days', 'dispatch_days', 'delivery_days',
       'expected_delivery_days', 'delay_days', 'delay_days_int', 'is_delayed',
       'delay_days_cat', 'main_category', 'sub_category'],
      dtype='str')

In [111]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95784 entries, 0 to 95783
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   index                     95784 non-null  int64  
 1   order_id                  95784 non-null  str    
 2   customer_id               95784 non-null  str    
 3   customer_unique_id        95784 non-null  str    
 4   customer_zip_code_prefix  95784 non-null  int64  
 5   customer_city             95784 non-null  str    
 6   customer_state            95784 non-null  str    
 7   customer_lat              95735 non-null  float64
 8   customer_lng              95735 non-null  float64
 9   total_items_count         95784 non-null  int64  
 10  seller_id                 95784 non-null  str    
 11  price                     95784 non-null  float64
 12  freight_value             95784 non-null  float64
 13  review_score              95784 non-null  int64  
 14  category         

In [112]:
df['review_score'].value_counts()

review_score
5    56682
4    18913
1     9312
3     7935
2     2942
Name: count, dtype: int64

In [113]:
df.isna().sum()

index                          0
order_id                       0
customer_id                    0
customer_unique_id             0
customer_zip_code_prefix       0
customer_city                  0
customer_state                 0
customer_lat                  49
customer_lng                  49
total_items_count              0
seller_id                      0
price                          0
freight_value                  0
review_score                   0
category                    1370
seller_zip_code_prefix         0
seller_city                    0
seller_state                   0
seller_lat                     0
seller_lng                     0
order_purchase_dayofweek       0
order_purchase_month           0
approved_days                  0
dispatch_days                  0
delivery_days                  0
expected_delivery_days         0
delay_days                     0
delay_days_int                 0
is_delayed                     0
delay_days_cat                 0
main_categ

### 위도 경도 활용해서 거리 파생변수 생성

In [114]:
# Haversine 거리 계산 함수 (지구 곡률을 반영한 두 좌표 간의 직선 거리 km)
def haversine_vectorize(lat1, lon1, lat2, lon2):
    # 라디안 변환
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c # 6371: 지구의 평균 반지름(km)
    return km

# 거리(distance_km) 파생 변수 생성
df['distance_km'] = haversine_vectorize(
    df['customer_lat'], df['customer_lng'],
    df['seller_lat'], df['seller_lng']
)

print(f"distance_km 결측치 제거 전 행 수 : {len(df)}")
# 결측치 처리 (제거)
# 일부 판매자 우편번호가 매핑 테이블에 없어 거리가 NaN으로 나올 수 있음
missing_distance = df['distance_km'].isnull().sum()
print(f"좌표 매핑 불가로 인한 거리 결측치 수: {missing_distance}건 (전체의 약 {missing_distance/len(df)*100:.1f}%) -> 제거")
df = df.dropna(subset=['distance_km']).reset_index()

print(f"distance_km 결측치 제거 후 행 수 : {len(df)}")

# 모델 학습에 불필요한 중간 컬럼 삭제
cols_to_drop = ['customer_zip_code_prefix', 'seller_zip_code_prefix', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']
df = df.drop(columns=cols_to_drop)

distance_km 결측치 제거 전 행 수 : 95784
좌표 매핑 불가로 인한 거리 결측치 수: 49건 (전체의 약 0.1%) -> 제거
distance_km 결측치 제거 후 행 수 : 95735


In [115]:
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())

In [116]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95735 entries, 0 to 95734
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   level_0                   95735 non-null  int64  
 1   index                     95735 non-null  int64  
 2   order_id                  95735 non-null  str    
 3   customer_id               95735 non-null  str    
 4   customer_unique_id        95735 non-null  str    
 5   customer_city             95735 non-null  str    
 6   customer_state            95735 non-null  str    
 7   total_items_count         95735 non-null  int64  
 8   seller_id                 95735 non-null  str    
 9   price                     95735 non-null  float64
 10  freight_value             95735 non-null  float64
 11  review_score              95735 non-null  int64  
 12  category                  94365 non-null  str    
 13  seller_city               95735 non-null  str    
 14  seller_state     

In [117]:
# 1. 독립변수(X)와 종속변수(y) 분리
# 분석 목적에 따라 배송 완료 전 알 수 있는 정보 위주로 구성
X = df[['total_items_count', 'main_category', 'sub_category', 
        'approved_days', 'dispatch_days', 'delivery_days', 'expected_delivery_days', 
        'delay_days', 'is_delayed', 'seller_state', 'customer_state']]
y = df['review_score']
y_binary = y.apply(lambda x: 0 if x <= 3 else 1)

# 2. 전처리 프로세스 정의
numeric_features = ['total_items_count', 'approved_days', 'dispatch_days', 'delivery_days', 'expected_delivery_days', 'delay_days']
categorical_features = ['main_category', 'sub_category', 'is_delayed', 'seller_state', 'customer_state']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 3. 모델 및 전체 파이프라인 구축
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

# 4. 데이터 분할 (Train/Test)
# 시계열 데이터일 경우 시간 순 분할이 권장되나, 일반적인 경우 무작위 분할 수행
X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=0.2, random_state=42)

# 5. 모델 학습
model.fit(X_train, y_train)

# 6. 결과 확인
y_pred = model.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           0       0.58      0.25      0.35      3197
           1       0.83      0.95      0.89     12121

    accuracy                           0.80     15318
   macro avg       0.70      0.60      0.62     15318
weighted avg       0.78      0.80      0.77     15318



In [118]:
# train / test / validation split 

X = df[cols]
y = df['review_score']

# Train+Valid / Test 분리
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

KeyError: "['customer_zip_code_prefix', 'customer_lat', 'customer_lng', 'seller_zip_code_prefix', 'seller_lat', 'seller_lng'] not in index"

In [ ]:
df2 = df.copy()

df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 95784 entries, 0 to 95783
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   index                     95784 non-null  int64  
 1   order_id                  95784 non-null  str    
 2   customer_id               95784 non-null  str    
 3   customer_unique_id        95784 non-null  str    
 4   customer_zip_code_prefix  95784 non-null  int64  
 5   customer_city             95784 non-null  str    
 6   customer_state            95784 non-null  str    
 7   customer_lat              95521 non-null  float64
 8   customer_lng              95521 non-null  float64
 9   total_items_count         95784 non-null  int64  
 10  seller_id                 95784 non-null  str    
 11  price                     95784 non-null  float64
 12  freight_value             95784 non-null  float64
 13  review_score              95784 non-null  int64  
 14  category         

In [ ]:
# 1. 그룹화 함수 정의
def group_score(score):
    if score >= 4: return 2   # High (긍정)
    elif score == 3: return 1 # Mid (보통)
    else: return 0            # Low (부정)

# 2. 타겟 변수 재구조화
y_group = df2['review_score'].apply(group_score)

# 3. 데이터 분할 및 학습
X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y_group, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=0.2, random_state=42)

# 모델 학습
model_group = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=15, random_state=42, n_jobs=-1)
model_group.fit(X_train, y_train)

# 검증 세트 점수 확인 (튜닝용)
val_score = model_group.score(X_valid, y_valid)
print(f"검증 세트 정확도: {val_score:.4f}")

# 최종 테스트 세트 점수 확인 (최종 성능)
test_score = model_group.score(X_test, y_test)
print(f"최종 테스트 세트 정확도: {test_score:.4f}")

ValueError: could not convert string to float: '83eeee55d222c6e3b10f716700517529'

In [ ]:
# train / test / validation split 
features = ['total_items_count', 'is_delayed', 'customer_state', 
            'delivery_days', 'dispatch_days', 'approved_days']
X = pd.get_dummies(df2[features], drop_first=True)
y = df2['review_score'].astype(int)

# Train+Valid / Test 분리
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)
# Train / Valid 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.2, random_state=42,
)

컬럼명: Index(['index', 'order_id', 'customer_id', 'customer_city', 'customer_state',
       'total_items_count', 'seller_id', 'review_score', 'category',
       'seller_city', 'seller_state', 'order_purchase_dayofweek',
       'order_purchase_month', 'approved_days', 'dispatch_days',
       'delivery_days', 'expected_delivery_days', 'delay_days',
       'delay_days_int', 'is_delayed', 'delay_days_cat', 'main_category',
       'sub_category'],
      dtype='str')


In [ ]:
# 3) 교차검증으로 최적 차수 탐색
cv = KFold(n_splits=5, shuffle=True, random_state=42)

degrees = [1, 2, 3] # 4 이상은 현실적으로 무리..
cv_scores = []

for d in degrees:
    # 파이프라인 구축: 스케일링은 다항 변환 전후 어디든 중요하지만, 보통 전후로 챙깁니다.
    pipeline = Pipeline([
        ('scaler', StandardScaler()), 
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('model', LinearRegression())
    ])
    
    # 교차 검증
    score = cross_val_score(pipeline, X, y, cv=cv, scoring="r2").mean()
    cv_scores.append(score)

# 결과 확인
print(cv_scores) 

[np.float64(0.17502596470770732), np.float64(0.18702529410646396), np.float64(-0.14357439575467865)]


In [ ]:

# 모델 학습
model = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=10, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(